# ferric quickstart: an energy and its certificate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mgoldey/ferric/blob/main/examples/notebooks/01_quickstart.ipynb)

Hartree-Fock and RI-MP2 on water, and the validation certificate every result carries.

Runs on a Colab CPU runtime in about a minute. Every number printed below is also asserted, so a cell that runs without an error has reproduced the reference value.

In [ ]:
%pip install -q --pre ferric py3Dmol
import os

os.environ.setdefault(
    "OPENBLAS_NUM_THREADS", "1"
)  # set before the first `import ferric`

In [ ]:
import ferric

try:
    from ferric import viz
except ImportError as err:
    raise ImportError(
        "this notebook needs a ferric release that ships ferric.viz "
        "(newer than 0.1.0rc11)"
    ) from err
print(ferric.__build__)

## A geometry

`Molecule.from_xyz_string` reads Angstrom; `coords()` gives Angstrom back.

In [ ]:
WATER_XYZ = """3
water
O   0.000000   0.000000   0.117790
H   0.000000   0.755453  -0.471161
H   0.000000  -0.755453  -0.471161
"""
water = ferric.Molecule.from_xyz_string(WATER_XYZ, 0, 1)  # charge 0, singlet
viz.show_molecule(water)

## Hartree-Fock in STO-3G

**Always read `converged`**: a run that hits its iteration limit still returns an energy and sets the flag instead of raising. Energies are in Hartree.

In [ ]:
sto3g = ferric.BasisSet.bundled("sto-3g")
rhf = ferric.run_rhf(water, sto3g)
print(rhf.converged, f"{rhf.energy:.10f}")

# Reference: site/src/using/quickstart.md (checked against PySCF to 1e-8 Ha).
assert rhf.converged
assert abs(rhf.energy - (-74.9631468000)) < 1e-7

## The certificate

Every result carries a certificate: the validation grade of the method that produced it, the build, and the approximations the run used. An unconverged SCF is graded `ungraded`. The grades are defined in [Capabilities and validation](https://mgoldey.github.io/ferric/reference/validation.html).

In [ ]:
cert = rhf.certificate
print("capability:", cert["capability"])
print("grade     :", cert["grade"], "(from", cert["grade_source"] + ")")
print("build     :", cert["build"])
print("approximations:", cert["approximations"])
print("warnings  :", cert["warnings"])
assert cert["grade"] != "ungraded"

## Correlation: RI-MP2 in cc-pVDZ

RI-MP2 needs an auxiliary (fitting) basis next to the orbital basis; STO-3G is too small for a meaningful correlation energy.

In [ ]:
bs = ferric.BasisSet.bundled("cc-pvdz")
aux = ferric.BasisSet.bundled("cc-pvdz-ri")
mp2 = ferric.run_rimp2(water, bs, aux)
print(f"RI-MP2 total energy: {mp2.total_energy:.10f} Ha")
print(
    "grade:",
    mp2.certificate["grade"],
    "| approximations:",
    [a["name"] for a in mp2.certificate["approximations"]],
)

# Reference: site/src/using/quickstart.md.
assert abs(mp2.total_energy - (-76.2308014541)) < 1e-6

The same calculations run from the command line (`ferric water.toml`). See [Your first calculation](https://mgoldey.github.io/ferric/using/quickstart.html) and the other notebooks in this folder.